# Chapter 4 — Functions and Scope

**Defining functions, parameter passing, closures, and the LEGB scope model.**

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HusseinYounis/Python/blob/main/public/notebooks/ch04/04_functions_and_scope.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/HusseinYounis/Python/main?filepath=public/notebooks/ch04/04_functions_and_scope.ipynb)

> Click a badge above to run this notebook in your browser — no installation needed.

## Learning Outcomes

- Define functions and understand how Python differs from C++ in return types, argument types, and duck typing.
- Use positional, keyword, default, `*args`, `**kwargs`, keyword-only, and positional-only parameters.
- Explain the LEGB scope resolution rule, `global`, `nonlocal`, and the UnboundLocalError trap.
- Write lambda expressions and use them with `sorted()`, `map()`, and `filter()`.
- Create closures, higher-order functions, and understand decorators.
- Add type hints, enforce types at runtime with `isinstance()`, and use `mypy` for static checking.

---
## 4.1 Defining Functions

Functions are **first-class objects** in Python — defined at runtime, no type declarations required, and can be passed around like any other value.

In [ ]:
def greet(name):
    """Return a greeting string."""
    return f"Hello, {name}!"

print(greet("World"))
print(greet(42))           # No type restriction — duck typing!
print(type(greet))

### Duck Typing — Same Function, Multiple Types

In [ ]:
# No return type, no param types needed
def add(a, b):
    return a + b

# Same function works with ANY type that supports +
print(f"int:   {add(3, 5)}")
print(f"float: {add(1.5, 2.5)}")
print(f"str:   {add('hi ', 'there')}")
print(f"list:  {add([1, 2], [3, 4])}")

### Multiple Return Values

In [ ]:
def divide_and_remainder(a, b):
    return a // b, a % b   # returns a tuple

quotient, remainder = divide_and_remainder(17, 5)
print(f"17 ÷ 5 = {quotient} remainder {remainder}")

### Functions as Objects

In [ ]:
# Assign to a variable
say_hello = greet
print(say_hello("Alice"))

# Store in a data structure
operations = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
}
print(f"add(10, 3) = {operations['add'](10, 3)}")

# Pass as argument
def apply_twice(func, value):
    return func(func(value))

print(f"apply_twice(double, 3) = {apply_twice(lambda x: x * 2, 3)}")

---
## 4.2 Parameter Types

Python offers far more flexible parameter passing than C++.

### Positional, Keyword, and Default Arguments

In [ ]:
def connect(host, port=3306, timeout=30):
    print(f"Connecting to {host}:{port} (timeout={timeout}s)")

connect("db.server.com")                    # uses defaults
connect("db.server.com", 5432)              # port=5432
connect("db.server.com", timeout=10)        # keyword arg

### `*args` and `**kwargs`

In [ ]:
# *args — collects extra positional args into a tuple
def total(*values):
    print(f"type: {type(values)}, values: {values}")
    return sum(values)

print(f"total(1,2,3) = {total(1, 2, 3)}")
print(f"total(10,20,30,40) = {total(10, 20, 30, 40)}")

In [ ]:
# **kwargs — collects extra keyword args into a dict
def build_config(**kwargs):
    print(f"type: {type(kwargs)}")
    for key, value in kwargs.items():
        print(f"  {key} = {value}")

build_config(debug=True, port=8080, host="localhost")

In [ ]:
# Combining all parameter types
def mega_func(a, b, *args, option=True, **kwargs):
    print(f"a={a}, b={b}")
    print(f"args={args}")
    print(f"option={option}")
    print(f"kwargs={kwargs}")

mega_func(1, 2, 3, 4, 5, option=False, x=10, y=20)

### Keyword-Only and Positional-Only Parameters

In [ ]:
# Keyword-only: parameters after * must be passed by name
def safe_div(a, b, *, round_digits=2):
    return round(a / b, round_digits)

print(safe_div(10, 3, round_digits=4))
# safe_div(10, 3, 4)  # TypeError!

# Positional-only: parameters before / must be passed by position
def power(base, exp, /):
    return base ** exp

print(power(2, 10))
# power(base=2, exp=10)  # TypeError!

### Unpacking Arguments with `*` and `**`

In [ ]:
def point(x, y, z):
    print(f"({x}, {y}, {z})")

coords = [1, 2, 3]
point(*coords)           # Unpack list

config = {"x": 10, "y": 20, "z": 30}
point(**config)          # Unpack dict

### The Mutable Default Trap

In [ ]:
# BUG — the list persists between calls
def append_to_buggy(item, target=[]):
    target.append(item)
    return target

print("Buggy:")
print(append_to_buggy(1))   # [1]
print(append_to_buggy(2))   # [1, 2] — BUG!
print(append_to_buggy(3))   # [1, 2, 3] — keeps growing!

# FIX — use None as sentinel
def append_to_fixed(item, target=None):
    if target is None:
        target = []
    target.append(item)
    return target

print("\nFixed:")
print(append_to_fixed(1))   # [1]
print(append_to_fixed(2))   # [2] ✓

---
## 4.3 Scope and the LEGB Rule

Python resolves names in this order:
- **L**ocal — inside the current function
- **E**nclosing — outer function scopes
- **G**lobal — module (file) level
- **B**uilt-in — `print`, `len`, etc.

In [ ]:
x = "global"       # G

def outer():
    x = "enclosing"  # E
    def inner():
        x = "local"   # L
        print(f"inner: {x}")
    inner()
    print(f"outer: {x}")

outer()
print(f"module: {x}")

### `global` and `nonlocal`

In [ ]:
# global — modify a module-level variable
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
print(f"counter = {counter}")

In [ ]:
# nonlocal — modify an enclosing function's variable
def make_counter():
    count = 0
    def inc():
        nonlocal count
        count += 1
        return count
    return inc

c = make_counter()
print(c())   # 1
print(c())   # 2
print(c())   # 3

### No Block Scope in Python!

In [ ]:
# Variables from loops and if-blocks are still accessible!
for i in range(5):
    message = "inside loop"

print(f"i = {i}")              # 4 — still accessible!
print(f"message = {message}")  # still accessible!

---
## 4.4 Lambda Expressions

Small **anonymous** functions limited to a **single expression**.

In [ ]:
# Basic lambda
square = lambda x: x ** 2
print(f"square(5) = {square(5)}")

add = lambda a, b: a + b
print(f"add(3, 7) = {add(3, 7)}")

### Common Use: Sorting

In [ ]:
# Sort students by GPA (descending)
students = [("Ali", 3.5), ("Sara", 3.9), ("Omar", 3.7)]
ranked = sorted(students, key=lambda s: s[1], reverse=True)
print("By GPA:", ranked)

# Sort strings by length
words = ["python", "is", "awesome", "and", "powerful"]
print("By length:", sorted(words, key=lambda w: len(w)))

### Lambda with `map()` and `filter()`

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Double each number
doubled = list(map(lambda x: x * 2, numbers))
print(f"Doubled: {doubled}")

# Keep only even numbers
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(f"Evens: {evens}")

# Same as comprehension (often clearer)
result = [x * 2 for x in numbers if x % 2 == 0]
print(f"Comprehension: {result}")

---
## 4.5 Closures and Higher-Order Functions

In [ ]:
# Closure — function that remembers enclosing scope
def make_multiplier(factor):
    def multiplier(x):
        return x * factor
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(f"double(5) = {double(5)}")
print(f"triple(5) = {triple(5)}")

In [ ]:
# Practical closure: logger
def make_logger(prefix):
    def log(message):
        print(f"[{prefix}] {message}")
    return log

info = make_logger("INFO")
error = make_logger("ERROR")

info("Server started")
error("Disk full")

### Built-in Higher-Order Functions + `reduce()`

In [ ]:
from functools import reduce

nums = [1, 2, 3, 4, 5]

squared = list(map(lambda x: x ** 2, nums))
print(f"map:    {squared}")

evens = list(filter(lambda x: x % 2 == 0, nums))
print(f"filter: {evens}")

total = reduce(lambda a, b: a + b, nums)
print(f"reduce: {total}")

### Decorators (Preview)

In [ ]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

slow_sum(1_000_000)

---
## 4.6 Type Hints and Enforcing Types

Type hints are **optional documentation** — Python does NOT enforce them at runtime.

In [ ]:
# Function with type hints
def add_typed(a: int, b: int) -> int:
    return a + b

# Hints are NOT enforced!
print(add_typed(3, 5))               # 8 — works
print(add_typed("hello", " world"))  # still works!

In [ ]:
# Enforcing types at runtime with isinstance()
def add_safe(a, b):
    if not isinstance(a, (int, float)):
        raise TypeError(f"Expected int or float, got {type(a).__name__}")
    if not isinstance(b, (int, float)):
        raise TypeError(f"Expected int or float, got {type(b).__name__}")
    return a + b

print(add_safe(3, 5))       # 8
print(add_safe(1.5, 2.5))   # 4.0

try:
    add_safe("hello", "world")
except TypeError as e:
    print(f"Error: {e}")

### Docstrings

In [ ]:
def binary_search(arr: list[int], target: int) -> int:
    """Return the index of target in a sorted list, or -1 if not found.

    Args:
        arr: A sorted list of integers.
        target: The integer to search for.

    Returns:
        The index of target, or -1 if not found.
    """
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

data = [1, 3, 5, 7, 9, 11, 13]
print(f"Search 7:  index {binary_search(data, 7)}")
print(f"Search 4:  index {binary_search(data, 4)}")

# View the docstring
help(binary_search)

---
## Exercises

Complete the exercises below directly in this notebook!

### Exercise 1
Write a function `clamp(value, *, lo=0, hi=100)` that returns the value bounded within `[lo, hi]`, using keyword-only arguments.

In [ ]:
# Exercise 1: Your code here
def clamp(value, *, lo=0, hi=100):
    pass  # Replace with your implementation

# Test
print(clamp(150))              # 100
print(clamp(-10))              # 0
print(clamp(50))               # 50
print(clamp(5, lo=1, hi=10))   # 5

### Exercise 2
Write a function `describe(*args, **kwargs)` that prints how many positional and keyword arguments it received.

In [ ]:
# Exercise 2: Your code here
def describe(*args, **kwargs):
    pass  # Replace with your implementation

# Test
describe(1, 2, 3, name="Alice", age=25)

### Exercise 3
Create a `make_counter()` closure that returns `increment` and `reset` functions sharing the same `count` variable.

In [ ]:
# Exercise 3: Your code here
def make_counter():
    pass  # Replace: return increment, reset functions

# Test
# inc, reset = make_counter()
# print(inc())    # 1
# print(inc())    # 2
# reset()
# print(inc())    # 1

### Exercise 4
Use `sorted()` with a lambda to sort a list of dictionaries by `"age"`.

In [ ]:
# Exercise 4: Your code here
people = [
    {"name": "Ali", "age": 20},
    {"name": "Sara", "age": 18},
    {"name": "Omar", "age": 22},
]

# Sort by age

### Exercise 5
Use `map()` to convert a list of Celsius temperatures to Fahrenheit (`F = C * 9/5 + 32`).

In [ ]:
# Exercise 5: Your code here
celsius = [0, 20, 37, 100]

# Convert with map and lambda

### Exercise 6
Write a function that returns `min`, `max`, `mean`, and `count` for a list of numbers. Add proper type hints and a docstring.

In [ ]:
# Exercise 6: Your code here
def stats(numbers):
    pass  # Return (min, max, mean, count)

# Test
data = [10, 20, 30, 40, 50]
# lo, hi, avg, n = stats(data)
# print(f"min={lo}, max={hi}, mean={avg}, count={n}")

---

**Next Chapter:** [Chapter 5 — Data Structures](../ch05/)